In [0]:
"""
01_machine_dimension.py

Machine Dimension (SCD Type 1)

Source:
    operation_events

Target:
    machine_dimension

Author:
Sumanth Vempalle

Version:
2.2.0
"""

import dlt

from pyspark.sql.functions import (
    col,
)

# ============================================================
# Machine Source View
# ============================================================

@dlt.view(
    name="machine_dimension_source",
    comment="Source view for Machine Dimension."
)
def machine_dimension_source():

    return (

        spark.readStream.table(
            "operation_events"
        )

        .select(

            col("machine_id"),

            col("machine_name"),

            col("machine_type"),

            col("station_code"),

            col("station_type"),

            col("line_id"),

            col("line_name"),

            col("hall_id"),

            col("hall_name"),

            col("department"),

            col("event_timestamp")

                .alias("last_updated"),

        )

        .dropDuplicates(
            ["machine_id"]
        )

    )


# ============================================================
# Target Streaming Table
# ============================================================

dlt.create_streaming_table(

    name="machine_dimension",

    comment="Machine Dimension (SCD Type 1)."

)


# ============================================================
# AUTO CDC FLOW
# ============================================================

dlt.create_auto_cdc_flow(

    target="machine_dimension",

    source="machine_dimension_source",

    keys=[

        "machine_id",

    ],

    sequence_by="last_updated",

    stored_as_scd_type=1,

)